# 5. Graph API vs. Functional API — When to Use Which

LangGraph offers **two ways to build the same underlying graph**, running on the same engine underneath:

| | Graph API (what we've used so far) | Functional API |
|---|---|---|
| Building blocks | `StateGraph`, `add_node`, `add_edge` | `@entrypoint`, `@task` |
| Control flow | Declared edges | Plain Python `if` / function calls |
| State | Explicit schema (`TypedDict`) + partial updates | Local variables, scoped to the function |

This notebook's job is to prevent confusion later when you encounter Functional API code elsewhere and wonder if it's a different library — it's the same engine, different surface syntax. We'll reimplement **the exact three-node Applicant Screener pipeline from Video 4**, node-for-node, so the comparison is apples-to-apples.

**Sections**
1. Recap: the utility from Video 3
2. The Video 4 pipeline, as a reminder (Graph API)
3. The same pipeline, Functional API
4. Confirming identical output
5. What's gained, what's lost
6. Decision guide — and why this course uses the Graph API from here on

## 1. Recap: the utility from Video 3

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for env_var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_API_KEY"]:
    if not os.environ.get(env_var):
        os.environ[env_var] = getpass.getpass(f"{env_var}: ")

In [ ]:
%%writefile llm_utils.py
"""Provider-agnostic LLM utility used by every node in this course from Video 3 onward.

Call `call_llm(provider, messages, system=None)` instead of ever importing a provider
SDK directly from graph/node code. Swapping models is then a one-line `provider` change.
"""
from dataclasses import dataclass

import anthropic
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

# Standardize on one current, non-legacy model per provider.
CLAUDE_MODEL = "claude-sonnet-5"
GPT_MODEL = "gpt-5.1"
GEMINI_MODEL = "gemini-3.5-flash"

DEFAULT_PROVIDER = "claude"  # the rest of this course leaves this default in place

_anthropic_client = anthropic.Anthropic()
_openai_client = OpenAI()
_gemini_client = genai.Client()


@dataclass
class LLMResponse:
    content: str
    provider: str
    model: str
    raw: object  # the untouched provider response, for debugging


def _call_claude(messages: list[dict], system: str | None) -> LLMResponse:
    resp = _anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system or anthropic.NOT_GIVEN,
        messages=messages,
    )
    # Newer Claude models may put a thinking block first, so join only the text blocks.
    text = "".join(block.text for block in resp.content if block.type == "text")
    return LLMResponse(content=text, provider="claude", model=CLAUDE_MODEL, raw=resp)


def _call_gpt(messages: list[dict], system: str | None) -> LLMResponse:
    full_messages = ([{"role": "system", "content": system}] if system else []) + messages
    resp = _openai_client.chat.completions.create(model=GPT_MODEL, messages=full_messages)
    return LLMResponse(content=resp.choices[0].message.content, provider="gpt", model=GPT_MODEL, raw=resp)


def _to_gemini_contents(messages: list[dict]) -> list[genai_types.Content]:
    role_map = {"user": "user", "assistant": "model"}
    return [
        genai_types.Content(role=role_map[m["role"]], parts=[genai_types.Part(text=m["content"])])
        for m in messages
    ]


def _call_gemini(messages: list[dict], system: str | None) -> LLMResponse:
    config = genai_types.GenerateContentConfig(system_instruction=system) if system else None
    resp = _gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=_to_gemini_contents(messages),
        config=config,
    )
    return LLMResponse(content=resp.text, provider="gemini", model=GEMINI_MODEL, raw=resp)


_PROVIDERS = {"claude": _call_claude, "gpt": _call_gpt, "gemini": _call_gemini}


def call_llm(provider: str = DEFAULT_PROVIDER, messages: list[dict] | None = None, system: str | None = None) -> LLMResponse:
    if provider not in _PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}, expected one of {list(_PROVIDERS)}")
    return _PROVIDERS[provider](messages or [], system)

In [ ]:
from typing import TypedDict
from IPython.display import Image, display

from llm_utils import call_llm

sample_resume = """
Jordan Lee
6 years of experience as a backend software engineer.
Skills: Python, distributed systems, PostgreSQL, Kubernetes, API design.
Previously at a mid-size fintech company leading a 4-person team.
"""

PARSE_SYSTEM_PROMPT = (
    "You extract structured fields from a resume. Reply with exactly three lines, "
    "no extra commentary:\n"
    "Name: <candidate name>\n"
    "Years: <years of experience as a number>\n"
    "Skills: <comma-separated skills>"
)

RUBRIC = (
    "Score the candidate from 0-100 against this fixed rubric:\n"
    "- 5+ years experience: up to 40 points\n"
    "- Skills overlap with [Python, distributed systems, Kubernetes, SQL]: up to 40 points\n"
    "- Leadership experience mentioned: up to 20 points\n"
    "Reply with exactly two lines, no extra commentary:\n"
    "Score: <integer 0-100>\n"
    "Rationale: <one sentence>"
)


def show_graph(graph):
    """Render a compiled graph. Falls back to Mermaid text if the PNG renderer is unavailable."""
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:
        print(graph.get_graph().draw_mermaid())

## 2. The Video 4 pipeline, as a reminder (Graph API)

Three linear nodes, an explicit `TypedDict` state schema, edges declared separately from the node logic. This is exactly what Video 4 built.

In [ ]:
from langgraph.graph import StateGraph, START, END


class ScreenerState(TypedDict):
    resume_text: str
    candidate_name: str
    years_experience: str
    skills: str
    score: int
    score_rationale: str
    summary: str


def parse_resume(state: ScreenerState) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": state["resume_text"]}], system=PARSE_SYSTEM_PROMPT)
    fields = {"candidate_name": "", "years_experience": "", "skills": ""}
    for line in result.content.strip().splitlines():
        if line.startswith("Name:"):
            fields["candidate_name"] = line.removeprefix("Name:").strip()
        elif line.startswith("Years:"):
            fields["years_experience"] = line.removeprefix("Years:").strip()
        elif line.startswith("Skills:"):
            fields["skills"] = line.removeprefix("Skills:").strip()
    return fields


def score_against_rubric(state: ScreenerState) -> dict:
    candidate_summary = (
        f"Name: {state['candidate_name']}\n"
        f"Years of experience: {state['years_experience']}\n"
        f"Skills: {state['skills']}"
    )
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary}], system=RUBRIC)
    score, rationale = 0, ""
    for line in result.content.strip().splitlines():
        if line.startswith("Score:"):
            score = int("".join(ch for ch in line if ch.isdigit()) or 0)
        elif line.startswith("Rationale:"):
            rationale = line.removeprefix("Rationale:").strip()
    return {"score": score, "score_rationale": rationale}


def generate_summary(state: ScreenerState) -> dict:
    prompt = (
        f"Candidate: {state['candidate_name']}\n"
        f"Score: {state['score']}/100 ({state['score_rationale']})\n\n"
        "Write a two-sentence plain-text summary of this candidate for a hiring manager."
    )
    result = call_llm("claude", messages=[{"role": "user", "content": prompt}])
    return {"summary": result.content}


graph_builder = StateGraph(ScreenerState)
graph_builder.add_node("parse_resume", parse_resume)
graph_builder.add_node("score_against_rubric", score_against_rubric)
graph_builder.add_node("generate_summary", generate_summary)
graph_builder.add_edge(START, "parse_resume")
graph_builder.add_edge("parse_resume", "score_against_rubric")
graph_builder.add_edge("score_against_rubric", "generate_summary")
graph_builder.add_edge("generate_summary", END)

screener_graph = graph_builder.compile()
show_graph(screener_graph)

In [ ]:
graph_result = screener_graph.invoke({"resume_text": sample_resume})
graph_result

Notice what the Graph API made us write: an explicit **state schema**, **nodes** that return partial-update dicts, **edges** declared separately from the logic — and in exchange we got a **picture** of the pipeline for free (`show_graph` above).

## 3. The same pipeline, Functional API

Same three steps, same prompts, same `call_llm` utility — reimplemented with `@task` (one per step) and `@entrypoint` (the pipeline itself). No state schema, no edges: data just flows through function arguments and return values, like ordinary Python.

In [ ]:
from langgraph.func import entrypoint, task


@task
def parse_resume_task(resume_text: str) -> dict:
    result = call_llm("claude", messages=[{"role": "user", "content": resume_text}], system=PARSE_SYSTEM_PROMPT)
    fields = {"candidate_name": "", "years_experience": "", "skills": ""}
    for line in result.content.strip().splitlines():
        if line.startswith("Name:"):
            fields["candidate_name"] = line.removeprefix("Name:").strip()
        elif line.startswith("Years:"):
            fields["years_experience"] = line.removeprefix("Years:").strip()
        elif line.startswith("Skills:"):
            fields["skills"] = line.removeprefix("Skills:").strip()
    return fields


@task
def score_against_rubric_task(candidate_name: str, years_experience: str, skills: str) -> dict:
    candidate_summary = f"Name: {candidate_name}\nYears of experience: {years_experience}\nSkills: {skills}"
    result = call_llm("claude", messages=[{"role": "user", "content": candidate_summary}], system=RUBRIC)
    score, rationale = 0, ""
    for line in result.content.strip().splitlines():
        if line.startswith("Score:"):
            score = int("".join(ch for ch in line if ch.isdigit()) or 0)
        elif line.startswith("Rationale:"):
            rationale = line.removeprefix("Rationale:").strip()
    return {"score": score, "score_rationale": rationale}


@task
def generate_summary_task(candidate_name: str, score: int, score_rationale: str) -> str:
    prompt = (
        f"Candidate: {candidate_name}\nScore: {score}/100 ({score_rationale})\n\n"
        "Write a two-sentence plain-text summary of this candidate for a hiring manager."
    )
    return call_llm("claude", messages=[{"role": "user", "content": prompt}]).content


@entrypoint()
def screener_workflow(resume_text: str) -> dict:
    parsed = parse_resume_task(resume_text).result()                       # tasks return futures -> .result()
    scored = score_against_rubric_task(
        parsed["candidate_name"], parsed["years_experience"], parsed["skills"]
    ).result()
    summary = generate_summary_task(parsed["candidate_name"], scored["score"], scored["score_rationale"]).result()
    return {**parsed, **scored, "summary": summary}

In [ ]:
func_result = screener_workflow.invoke(sample_resume)
func_result

Same behavior, but:
- **no state schema** — data flows through ordinary function arguments and return values,
- **no edges** — control flow is just calling one task after another,
- tasks return **futures**; call `.result()` to wait for them,
- **no static graph to draw** — the "graph" only exists at runtime, inferred from which tasks got called.

## 4. Confirming identical output

Both objects are the same underlying type under the hood, so both support `.invoke`, `.stream`, `.batch`, `.ainvoke`, etc. Let's line up the two results field by field.

In [ ]:
from langgraph.pregel import Pregel

print("compiled StateGraph is a Pregel:", isinstance(screener_graph, Pregel))
print("@entrypoint workflow is a Pregel:", isinstance(screener_workflow, Pregel))
print()

for field in ["candidate_name", "years_experience", "skills", "score"]:
    print(f"{field:>16}: graph={graph_result[field]!r}  |  func={func_result[field]!r}")

(Exact scores/rationale wording can vary slightly between runs since both versions call the same non-deterministic LLM — the point is the *pipeline behavior* is identical, not that every token matches.)

## 5. What's gained, what's lost

**Gained with the Functional API, for a pipeline this simple:**
- Noticeably less boilerplate — no schema class, no `add_edge` calls.
- Reads like ordinary sequential Python, which lowers the barrier for anyone on the team who hasn't learned the Graph API's vocabulary.

**Lost, or harder:**
- No `show_graph`-style visualization — there's no static graph object to draw until the function actually runs.
- Fine-grained routing and parallel fan-out (coming in later videos) are more naturally expressed with explicit conditional/parallel edges than with `if`s and futures scattered through a function body.
- State management differs in flavor, not just syntax: the Functional API leans on function arguments/return values and closures, not an explicit shared state object you can inspect as a single thing at any point.

Both versions get the **same checkpointing/persistence guarantees** under the hood (real depth on that in a later video) — that part of the engine doesn't care which API you used to build the graph.

## 6. Decision guide — and why this course uses the Graph API from here on

| If you need… | Pick |
|---|---|
| A diagram of the workflow (docs, debugging) | **Graph API** |
| Branching, looping, parallel fan-out/fan-in, subgraphs, multi-agent structure | **Graph API** |
| Fine-grained control over routing | **Graph API** |
| A straightforward, mostly-linear pipeline, minimal ceremony | **Functional API** |
| To add graph-style persistence to an existing, mostly-linear script fast | **Functional API** |

**Rule of thumb:** if you're sketching boxes and arrows in your head, reach for the Graph API. If you're narrating a script top to bottom, the Functional API will feel more natural.

Nearly everything remaining in this course — conditional edges, loops, parallel fan-out/fan-in, subgraphs, multi-agent supervisors — is Graph-API-native. **From here on, this course uses the Graph API exclusively**, so don't expect `@entrypoint`/`@task` to reappear; this was the one dedicated comparison.